# Upcoming API events and metadata

This notebook calls the live `GET /events` calendar endpoint and shows the metadata returned by the Explaining Markets API. It loads `EM_API_KEY` from the repository `.env` file but never prints the key, request headers, or `.env` contents.

The API generates the event metadata. This notebook only requests it, inspects its structure, and creates a convenient flattened view.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json
import os

import httpx
from dotenv import load_dotenv
from IPython.display import HTML, JSON, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Load values into the process environment. Do not display the return values.
load_dotenv(REPO_ROOT / ".env", override=False)

API_KEY = os.getenv("EM_API_KEY")
API_BASE_URL = os.getenv(
    "EM_API_BASE_URL",
    "https://api.explainingmarkets.ai/v1",
).rstrip("/")

assert API_KEY, "EM_API_KEY is not available. Check the repository .env file."
print("EM_API_KEY loaded: yes")
print("API base URL:", API_BASE_URL)

## 1. Choose the calendar window

The API quickstart documents optional `start_date` and `end_date` filters. Adjust `DAYS_AHEAD` to widen or narrow the upcoming-event window.

In [ ]:
DAYS_AHEAD = 14
today = datetime.now(timezone.utc).date()
start_date = today.isoformat()
end_date = (today + timedelta(days=DAYS_AHEAD)).isoformat()

request_params = {
    "start_date": start_date,
    "end_date": end_date,
}
request_params

## 2. Request the live calendar

Authentication uses the same `X-API-Key` header as the prediction client. The header is intentionally never displayed.

In [ ]:
with httpx.Client(timeout=30.0) as client:
    response = client.get(
        f"{API_BASE_URL}/events",
        params=request_params,
        headers={"X-API-Key": API_KEY},
    )

response.raise_for_status()
payload = response.json()
print("HTTP status:", response.status_code)
print("Top-level response type:", type(payload).__name__)

## 3. Normalize the response envelope

This accepts either a direct event list or an object containing an `events`, `items`, or `data` list so the inspection remains useful if the endpoint uses an envelope.

In [ ]:
if isinstance(payload, list):
    events = payload
elif isinstance(payload, dict):
    events = next(
        (payload[key] for key in ("events", "items", "data") if isinstance(payload.get(key), list)),
        None,
    )
    if events is None:
        raise ValueError(f"Could not find an event list. Top-level keys: {sorted(payload)}")
else:
    raise TypeError(f"Unexpected response type: {type(payload).__name__}")

print(f"Received {len(events)} events from {start_date} through {end_date}.")

## 4. Inspect the raw metadata

The first display shows one complete API event. The second lists every field observed across the returned events and the Python value types seen for it.

In [ ]:
if events:
    display(JSON(events[0], expanded=True))
else:
    print("No events were returned for this date window. Increase DAYS_AHEAD and rerun.")

In [ ]:
field_inventory = {}
for event in events:
    for key, value in event.items():
        field_inventory.setdefault(key, set()).add(type(value).__name__)

for field in sorted(field_inventory):
    print(f"{field:28} {', '.join(sorted(field_inventory[field]))}")

## 5. Flatten the useful scheduling metadata

This view does not generate new source metadata. It derives `tickers`, `hours_until_event`, and `hours_between_cutoff_and_event` from the API fields to make cache scheduling easier to inspect.

In [ ]:
def parse_utc(value):
    if not value:
        return None
    parsed = datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    return parsed.replace(tzinfo=parsed.tzinfo or timezone.utc)


def event_time(event):
    for key in ("event_datetime", "scheduled_at", "event_time", "timestamp"):
        if event.get(key):
            return parse_utc(event[key])
    return None


def focal_tickers(event):
    assets = event.get("focal_assets") or []
    return ", ".join(
        str(asset.get("identifier_value", ""))
        for asset in assets
        if isinstance(asset, dict)
    )


now = datetime.now(timezone.utc)
rows = []
for event in events:
    scheduled = event_time(event)
    cutoff = parse_utc(event.get("knowledge_cutoff"))
    rows.append({
        "event_id": event.get("event_id") or event.get("id"),
        "event_type": event.get("event_type"),
        "timing_category": event.get("timing_category") or event.get("status"),
        "tickers": focal_tickers(event),
        "event_datetime": scheduled.isoformat() if scheduled else None,
        "knowledge_cutoff": cutoff.isoformat() if cutoff else None,
        "hours_until_event": round((scheduled - now).total_seconds() / 3600, 1) if scheduled else None,
        "hours_between_cutoff_and_event": round((scheduled - cutoff).total_seconds() / 3600, 1) if scheduled and cutoff else None,
    })

rows.sort(key=lambda row: row["event_datetime"] or "")
rows[:5]

In [ ]:
if rows:
    columns = list(rows[0])
    header = "".join(f"<th>{column}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{row.get(column, '')}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<div style='overflow:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))
else:
    print("No rows to display.")

## 6. Identify events approaching T-36h

A job running every six hours can select events between 30 and 42 hours away, bracketing the T-36h target. This cell shows which currently returned events fall in that window.

In [ ]:
cache_due = [
    row for row in rows
    if row["hours_until_event"] is not None
    and 30 <= row["hours_until_event"] <= 42
]

print(f"{len(cache_due)} events currently fall in the T-36h cache window.")
cache_due

## 7. Identify events during the competition window

Knowing which companies will release earnings allows for industry mappings, etc.

In [21]:
start_date = "2026-08-10"
end_date = "2026-09-30"

request_params = {
    "start_date": start_date,
    "end_date": end_date,
}

In [22]:
competition_start = datetime(
    2026, 8, 10,
    tzinfo=timezone.utc,
)

competition_end = datetime(
    2026, 10, 1,
    tzinfo=timezone.utc,
)

competition_events = [
    event
    for event in events
    if (
        event_time(event) is not None
        and competition_start
        <= event_time(event)
        < competition_end
    )
]

print(f"{len(competition_events)} competition events")

1093 competition events


In [23]:
ticker_rows = []

for event in competition_events:
    scheduled = event_time(event)

    for asset in event.get("focal_assets", []):
        if asset.get("identifier_type") != "TICKER":
            continue

        ticker_rows.append(
            {
                "event_id": event.get("event_id") or event.get("id"),
                "ticker": asset.get("identifier_value"),
                "event_datetime": (
                    scheduled.isoformat()
                    if scheduled
                    else None
                ),
                "event_type": event.get("event_type"),
                "timing_category": event.get("timing_category"),
                "knowledge_cutoff": event.get("knowledge_cutoff"),
            }
        )

len(ticker_rows), ticker_rows[:5]

(1093,
 [{'event_id': 'ea_INTT_Q2_2026',
   'ticker': 'INTT',
   'event_datetime': '2026-08-10T10:15:00+00:00',
   'event_type': 'EARNINGS_RELEASE',
   'timing_category': 'SCHEDULED',
   'knowledge_cutoff': '2026-08-07T20:00:00Z'},
  {'event_id': 'ea_AIRS_Q2_2026',
   'ticker': 'AIRS',
   'event_datetime': '2026-08-10T10:30:00+00:00',
   'event_type': 'EARNINGS_RELEASE',
   'timing_category': 'SCHEDULED',
   'knowledge_cutoff': '2026-08-07T20:00:00Z'},
  {'event_id': 'ea_VERU_Q3_2026',
   'ticker': 'VERU',
   'event_datetime': '2026-08-10T10:30:00+00:00',
   'event_type': 'EARNINGS_RELEASE',
   'timing_category': 'SCHEDULED',
   'knowledge_cutoff': '2026-08-07T20:00:00Z'},
  {'event_id': 'ea_FERG_Q2_2026',
   'ticker': 'FERG',
   'event_datetime': '2026-08-10T10:45:00+00:00',
   'event_type': 'EARNINGS_RELEASE',
   'timing_category': 'SCHEDULED',
   'knowledge_cutoff': '2026-08-07T20:00:00Z'},
  {'event_id': 'ea_TH_Q2_2026',
   'ticker': 'TH',
   'event_datetime': '2026-08-10T10:45:00+

In [24]:
import csv

output_path = REPO_ROOT / "knowledge" / "calendar" / "competition_events.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

columns = [
    "event_id",
    "ticker",
    "event_datetime",
    "event_type",
    "timing_category",
    "knowledge_cutoff",
]

with output_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(ticker_rows)

print(f"Saved {len(ticker_rows)} event/ticker rows to {output_path}")

Saved 1093 event/ticker rows to C:\Users\wfpin\Desktop\Projects\explaining-markets\knowledge\calendar\competition_events.csv
